# QuantLab — Torneo ML de predicciones (notebook de ejemplo)

Pipeline mínimo y **ejecutable** de punta a punta:

1. Descarga los datasets `train` y `validation` de la ronda desde el worker de QuantLab.
2. Entrena un modelo pequeño sobre las columnas `feature_*` para predecir `target`.
3. Genera `predictions.csv` con las columnas **`id,prediction`**, listo para subir en la pestaña **Enviar**.

> QuantLab es una herramienta de investigación. No es asesoría financiera ni recomendación de inversión.

## Requisitos

```bash
pip install pandas pyarrow scikit-learn requests
# opcional (si está, se usa): pip install xgboost
```

## 1. Configuración

- `WORKER_URL`: la misma URL que usa el frontend (`NEXT_PUBLIC_WORKER_URL`).
- `TOURNAMENT_ID`: el id del torneo ML (está en la URL del detalle: `/app/tournaments/<id>`).
- `TOKEN`: opcional. El listado de datasets es público; si tu despliegue lo protege, pega
  el `access_token` de Supabase (DevTools → Application → Local Storage).

In [ ]:
import os

WORKER_URL = os.environ.get("QUANTLAB_WORKER_URL", "http://localhost:8001").rstrip("/")
TOURNAMENT_ID = os.environ.get("QUANTLAB_TOURNAMENT_ID", "")  # <-- pega aquí el id del torneo
TOKEN = os.environ.get("QUANTLAB_TOKEN", "")                  # opcional
ROUND = None                                                   # None = ronda más reciente disponible

HEADERS = {"Authorization": f"Bearer {TOKEN}"} if TOKEN else {}
print("worker:", WORKER_URL)
print("torneo:", TOURNAMENT_ID or "(sin definir: complétalo antes de seguir)")

## 2. Descargar train / validation

`GET /ml/datasets?tournament_id=...` lista la ronda. Para cada dataset `train`/`validation`
se pide su URL pública con `GET /ml/datasets/{id}/download?kind=...` (mismo endpoint que usa
el botón **Descargar** de la web).

In [ ]:
import io
import requests
import pandas as pd


def listar_datasets(tournament_id: str, round_number=None) -> list[dict]:
    params = {}
    if tournament_id:
        params["tournament_id"] = tournament_id
    if round_number is not None:
        params["round_number"] = round_number
    r = requests.get(f"{WORKER_URL}/ml/datasets", params=params, headers=HEADERS, timeout=60)
    r.raise_for_status()
    return r.json().get("datasets", [])


def url_descarga(dataset: dict) -> str:
    """URL del parquet: usa download_url si viene, si no pregunta al endpoint."""
    if dataset.get("download_url"):
        return dataset["download_url"]
    r = requests.get(
        f"{WORKER_URL}/ml/datasets/{dataset['id']}/download",
        params={"kind": dataset["kind"]},
        headers=HEADERS,
        timeout=60,
    )
    r.raise_for_status()
    return r.json()["url"]


def leer_parquet(url: str) -> pd.DataFrame:
    r = requests.get(url, timeout=300)
    r.raise_for_status()
    return pd.read_parquet(io.BytesIO(r.content))


datasets = listar_datasets(TOURNAMENT_ID, ROUND)
for d in datasets:
    print(
        f"ronda {d['round_number']:>3} | {d['kind']:<10} | modo={d['mode']:<9} "
        f"| estado={d['status']:<8} | filas={d.get('row_count')} "
        f"| activos={d.get('n_assets')} eras={d.get('n_eras')} features={d.get('n_features')}"
    )
if not datasets:
    print("Sin datasets: revisa TOURNAMENT_ID / WORKER_URL.")

In [ ]:
por_kind = {}
for d in sorted(datasets, key=lambda x: x.get("round_number") or 0):
    por_kind[d["kind"]] = d  # queda la ronda más alta de cada kind

ds_train = por_kind.get("train")
ds_valid = por_kind.get("validation")
ds_live = por_kind.get("live")
assert ds_train is not None, "La ronda no tiene dataset de train."

train = leer_parquet(url_descarga(ds_train))
valid = leer_parquet(url_descarga(ds_valid)) if ds_valid else None

print("train:", train.shape)
print("validation:", None if valid is None else valid.shape)
train.head()

## 3. Features y target

Las features son las columnas `feature_*` que declara el propio dataset (`feature_cols`).
No se inventan columnas: si `feature_cols` no viene, se detectan por prefijo.

In [ ]:
features = ds_train.get("feature_cols") or [c for c in train.columns if c.startswith("feature_")]
features = [c for c in features if c in train.columns]
assert features, "No se encontraron columnas feature_*."

TARGET = "target"
assert TARGET in train.columns, f"El train no tiene columna '{TARGET}'. Columnas: {list(train.columns)[:20]}"

print(f"{len(features)} features. Ejemplo: {features[:5]}")

X = train[features].astype("float32").fillna(0.5)
y = train[TARGET].astype("float32")
mask = y.notna()
X, y = X[mask], y[mask]
print("entrenamiento:", X.shape)

## 4. Entrenar un modelo pequeño

Usa XGBoost si está instalado; si no, `GradientBoostingRegressor` de scikit-learn.
Modelo deliberadamente pequeño: el objetivo es un baseline honesto, no ganar la ronda.

In [ ]:
modelo = None
try:
    from xgboost import XGBRegressor

    modelo = XGBRegressor(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        n_jobs=-1,
        random_state=42,
    )
    print("Modelo: XGBRegressor")
except ImportError:
    from sklearn.ensemble import GradientBoostingRegressor

    modelo = GradientBoostingRegressor(
        n_estimators=120,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        random_state=42,
    )
    print("Modelo: GradientBoostingRegressor (sklearn)")

modelo.fit(X, y)
print("entrenado.")

## 5. Validar (correlación por era)

La métrica del torneo es la correlación media por era, así que se mide igual aquí.

In [ ]:
def corr_por_era(df: pd.DataFrame, col_pred: str, col_target: str = TARGET) -> pd.Series:
    col_era = "era" if "era" in df.columns else None
    if col_era is None:
        return pd.Series({"global": df[col_pred].corr(df[col_target], method="spearman")})
    return df.groupby(col_era).apply(
        lambda g: g[col_pred].corr(g[col_target], method="spearman")
    )


if valid is not None and TARGET in valid.columns:
    Xv = valid[features].astype("float32").fillna(0.5)
    valid = valid.copy()
    valid["prediction"] = modelo.predict(Xv)
    corrs = corr_por_era(valid.dropna(subset=[TARGET]), "prediction")
    print(f"corr media   : {corrs.mean():.4f}")
    print(f"desv. std    : {corrs.std():.4f}")
    print(f"sharpe corr  : {corrs.mean() / corrs.std():.3f}" if corrs.std() else "")
    print(f"consistencia : {(corrs > 0).mean():.2%} de eras positivas")
else:
    print("Sin validation con target: se omite la validación.")

## 6. Generar `predictions.csv`

El dataset `live` no se descarga (nunca se expone su parquet). Se predice sobre `validation`
para dejar el CSV con el formato exacto; cuando la ronda `live` esté abierta, usa el mismo
código sobre las filas que te entregue la ronda.

**Formato obligatorio:** exactamente dos columnas, `id` y `prediction`, sin nulos y numérica.

In [ ]:
base = valid if valid is not None else train
Xb = base[features].astype("float32").fillna(0.5)

col_id = "id" if "id" in base.columns else base.columns[0]
salida = pd.DataFrame({
    "id": base[col_id].astype(str).values,
    "prediction": modelo.predict(Xb).astype("float64"),
})

# Validación local idéntica a la del worker
assert list(salida.columns) == ["id", "prediction"], salida.columns
assert not salida["prediction"].isna().any(), "hay NaN en prediction"
assert len(salida) > 0, "CSV vacío"

salida.to_csv("predictions.csv", index=False)
print(f"predictions.csv escrito: {len(salida)} filas")
salida.head()

## 7. Enviar

Arrastra `predictions.csv` a la pestaña **Enviar** del torneo en QuantLab.
El envío reemplaza el anterior de la misma ronda, así que puedes iterar sin penalización.

Ideas para mejorar el baseline: neutralizar features, promediar varias semillas,
validar por eras contiguas (no aleatorias) y vigilar la consistencia además de la correlación.